# AI-Lake + Apache Spark

Spark reads AI-Lake tables as standard Iceberg tables — no AI-Lake plugin required
for tabular queries. This notebook uses **PySpark local[*]** mode, which runs
entirely inside the Jupyter container (no separate Spark cluster needed).

**What this notebook covers:**
1. Single SparkSession with Iceberg JAR pre-loaded
2. Direct Parquet read (plain binary for `embedding`)
3. Iceberg HadoopCatalog SQL — full SQL interface
4. Filtered scans and aggregations
5. Iceberg snapshot history

Sections 10-14 additionally load the **ailake Spark plugin** (`io.ailake.spark.AilakeNative`, built into this image) and make real `deleteWhere`/`evolveSchema`/`search`/`scan`/`searchText`/`compact`/`writeBatch` calls via py4j — not just Iceberg-standard reads.

In [ ]:
import os
import pathlib
from pyspark.sql import SparkSession

TABLE_PATH = os.environ.get('DEMO_TABLE_PATH', '/data/ailake_demo')
ICEBERG_JAR = os.environ.get('ICEBERG_JAR', '/opt/iceberg/iceberg-spark-runtime-3.5_2.12-1.7.0.jar')

parquet_files = sorted(str(p) for p in pathlib.Path(TABLE_PATH).rglob('*.parquet'))
print(f'Table path   : {TABLE_PATH}')
print(f'Parquet files: {len(parquet_files)}')
print(f'Iceberg JAR  : {ICEBERG_JAR}')

## 1. Create SparkSession (Iceberg pre-loaded)

In [ ]:
import os
os.environ.setdefault('SPARK_LOCAL_IP', '127.0.0.1')

# Verify the Iceberg JAR was downloaded during Docker build
if not os.path.exists(ICEBERG_JAR):
    raise FileNotFoundError(
        f"Iceberg JAR not found: {ICEBERG_JAR}\n"
        "Rebuild the container: docker compose build demo"
    )

# ailake Spark plugin (§10-14 below) — built into the image at AILAKE_SPARK_JAR;
# gracefully degrades to Iceberg-only sections (1-9) when not present.
AILAKE_SPARK_JAR = os.environ.get('AILAKE_SPARK_JAR', '/usr/local/lib/ailake-spark-plugin.jar')
AILAKE_NATIVE_LIB = os.environ.get('AILAKE_JNI_LIB', '/usr/local/lib/libailake_jni.so')
AILAKE_PLUGIN_AVAILABLE = os.path.exists(AILAKE_SPARK_JAR) and os.path.exists(AILAKE_NATIVE_LIB)
spark_jars = ICEBERG_JAR if not AILAKE_PLUGIN_AVAILABLE else f'{ICEBERG_JAR},{AILAKE_SPARK_JAR}'

# Put the Iceberg JAR on the JVM classpath before the Py4J gateway starts.
# spark.jars adds it at runtime, but SparkCatalog plugin resolution uses
# the classloader fixed at JVM launch — SPARK_CLASSPATH is read by
# PySpark's launch_gateway() before the JVM starts, so the class is
# available when Catalogs.load() runs.
os.environ['SPARK_CLASSPATH'] = spark_jars

# Stop any stale SparkSession — getOrCreate() silently returns an existing
# session without applying new configs (including spark.jars), which means
# the Iceberg catalog plugin would not be on the JVM classpath.
_active = SparkSession.getActiveSession()
if _active is not None:
    _active.stop()

builder = (
    SparkSession.builder
    .appName('ailake-spark-demo')
    .master('local[2]')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.jars', spark_jars)
    .config(
        'spark.sql.extensions',
        'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions',
    )
    .config('spark.sql.catalog.ailake', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.ailake.type', 'hadoop')
    .config('spark.sql.catalog.ailake.warehouse', f'file://{TABLE_PATH}')
)
if AILAKE_PLUGIN_AVAILABLE:
    builder = builder.config('spark.driver.extraJavaOptions', f'-Dailake.native.lib={AILAKE_NATIVE_LIB}')

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print(f'Spark {spark.version} — local[2] with Iceberg extensions')
print(f'ailake Spark plugin: {"loaded (" + AILAKE_SPARK_JAR + ")" if AILAKE_PLUGIN_AVAILABLE else "not built into this image — sections 10-14 will SKIP"}')

## 2. Direct Parquet read

In [ ]:
df = spark.read.parquet(*parquet_files)
print(f'Rows: {df.count()}')
print('Schema:')
df.printSchema()

In [ ]:
# embedding shows as binary — HNSW footer is invisible to Spark
df.select('text').show(5, truncate=80)

## 3. Iceberg HadoopCatalog (SQL interface)

In [ ]:
# Same session — catalog plugin already loaded on classpath
count = spark.sql(
    'SELECT count(*) AS n FROM ailake.default.table'
).collect()[0][0]
print(f'Iceberg SQL count: {count}')

In [ ]:
spark.sql("""
    SELECT text
    FROM ailake.default.table
    WHERE text LIKE '%vector search%'
    LIMIT 5
""").show(truncate=90)

## 4. Aggregations

In [ ]:
spark.sql("""
    SELECT
        count(*)               AS total_rows,
        avg(length(text))      AS avg_text_len,
        avg(octet_length(embedding)) AS avg_emb_bytes
    FROM ailake.default.table
""").show()

## 5. Iceberg snapshot history

In [ ]:
spark.sql(
    'SELECT snapshot_id, committed_at, operation, summary'
    ' FROM ailake.default.table.snapshots'
).show(truncate=False)

## 6. Iceberg time-travel — read at specific snapshot

Iceberg keeps full snapshot history. `VERSION AS OF <snapshot_id>` reads the
table exactly as it was when that snapshot was committed — even if rows were
added or deleted since.


In [ ]:
# List all snapshots
snaps = spark.sql(
    'SELECT snapshot_id, committed_at, operation FROM ailake.default.table.snapshots'
).collect()
print(f'Snapshots available: {len(snaps)}')
for s in snaps:
    print(f'  {s.snapshot_id}  {s.committed_at}  {s.operation}')


In [ ]:
if len(snaps) >= 1:
    first_snap_id = snaps[0].snapshot_id
    n_first = spark.sql(
        f'SELECT count(*) AS n FROM ailake.default.table VERSION AS OF {first_snap_id}'
    ).collect()[0][0]
    n_now   = spark.sql('SELECT count(*) AS n FROM ailake.default.table').collect()[0][0]
    print(f'Rows at first snapshot ({first_snap_id}): {n_first}')
    print(f'Rows at current HEAD                   : {n_now}')
else:
    print('Only one snapshot — commit a second to demo time-travel.')


## 7. Iceberg snapshot metadata and file manifests

Iceberg stores file-level statistics (record count, file size) in manifest files.
Spark exposes these as system tables with the `$` suffix.


In [ ]:
# Per-file statistics from the Iceberg manifest
spark.sql(
    "SELECT file_path, record_count, file_size_in_bytes,"
    " round(file_size_in_bytes / record_count, 1) AS bytes_per_row"
    " FROM ailake.default.table.files"
).show(truncate=False)


In [ ]:
# Table-level Iceberg properties (AI-Lake custom props stored here)
spark.sql('SHOW TBLPROPERTIES ailake.default.table').show(truncate=False)
# ailake.embedding-model will appear here when embedding_model= was set at write time


## 8. Embedding model tracking via Spark `SHOW TBLPROPERTIES`

When a table is written with `embedding_model=`, Spark exposes `ailake.embedding-model`
via `SHOW TBLPROPERTIES`. Useful for auditing which model version was used for each table.

In [ ]:
# Filter AI-Lake embedding model properties specifically
# Note: table.properties is not available; use SHOW TBLPROPERTIES instead
all_props = spark.sql('SHOW TBLPROPERTIES ailake.default.table')
emb_props = all_props.filter(all_props['key'].like('ailake.embedding%'))
if emb_props.count() > 0:
    emb_props.show(truncate=False)
else:
    print('(ailake.embedding-model not set on this table)')
    print('Write with: ailake.open_table(..., embedding_model="text-embedding-3-small")')


In [ ]:
# Sections 9-14 below still use spark — session stays open until the final cell.


## 9. Iceberg v3 partitioned table

`init_demo.py` creates `ailake_partitioned_v3` — a format_version=3 table partitioned by `topic_id` (identity transform).
Spark reads it identically via Iceberg path load; partition pruning applies automatically.


In [ ]:
import pathlib

PV3_PATH = TABLE_PATH.replace("ailake_demo", "ailake_partitioned_v3")
pv3_data_dir = pathlib.Path(PV3_PATH) / "default" / "table"

# Direct Iceberg path read
try:
    pv3_df = spark.read.format("iceberg").load(f"file://{pv3_data_dir}")
    print(f"Rows: {pv3_df.count()}")
    pv3_df.printSchema()
    pv3_df.groupBy("topic_id").count().orderBy("topic_id").show(20)
except Exception as e:
    print(f"partitioned_v3 skipped: {e}")


## 10. `AilakeNative` py4j bridge — helpers

Sections 10-14 call `io.ailake.spark.AilakeNative` (a Scala `object`, exposed with
static forwarders) directly through py4j — the same JVM class the Fase 12 SQL/DataFrame
surface (`implicits.AilakeSession.ailakeSearch` etc., Scala-only) wraps. Raw py4j
reflection needs a few conversions Scala/py4j don't do automatically:
`scala.collection.Seq` (not a Java `List`), `scala.Option` (not nullable), and
primitive `float[]` (not a boxed list). `SKIP` prints below when the plugin jar
wasn't built into this image (see `AILAKE_PLUGIN_AVAILABLE` in §1).

In [ ]:
if not AILAKE_PLUGIN_AVAILABLE:
    print('SKIP — ailake Spark plugin not built into this image')
else:
    jvm = spark._jvm
    gw = spark.sparkContext._gateway
    AilakeNative = jvm.io.ailake.spark.AilakeNative

    def to_scala_seq(pylist):
        """Python list -> scala.collection.Seq, via java.util.ArrayList + JavaConverters."""
        jlist = jvm.java.util.ArrayList()
        for x in pylist:
            jlist.add(x)
        return jvm.scala.collection.JavaConverters.asScalaBuffer(jlist).toSeq()

    def none():
        """scala.None — py4j has no Scala Option support, Option.apply(null) gives None."""
        return jvm.scala.Option.apply(None)

    def to_float_array(pylist):
        """Python list[float] -> Java float[] (primitive array proxy — preserves type, unlike scalars)."""
        arr = gw.new_array(jvm.float, len(pylist))
        for i, v in enumerate(pylist):
            arr[i] = float(v)
        return arr

    def jfloat(v):
        return jvm.java.lang.Float(v)

    print('AilakeNative py4j helpers ready.')

## 11. `AilakeNative.deleteWhere` — Iceberg equality delete (real call)

Replaces the `print()`-only illustration from earlier revisions of this notebook —
this cell actually calls into the native library and commits an Iceberg equality
delete file, verified by a before/after row count from the AI-Lake plugin's own
`AilakeNative.search()` (not a separate Iceberg read).

In [ ]:
if not AILAKE_PLUGIN_AVAILABLE:
    print('SKIP')
else:
    DEL_PATH = TABLE_PATH.replace('ailake_demo', 'ailake_delete_demo')

    before = AilakeNative.search(DEL_PATH, to_float_array([0.0] * 32), 100, none(), none(), 'chunk_text', jfloat(0.5), 'default', 'table')
    print(f'Visible rows before Spark-plugin delete: {before.length()}')

    ids_to_delete = to_scala_seq(['20', '21'])
    ok = AilakeNative.deleteWhere(DEL_PATH, 'default', 'table', 'id', ids_to_delete)
    print(f"AilakeNative.deleteWhere('id', [20, 21]): {ok}")

    after = AilakeNative.search(DEL_PATH, to_float_array([0.0] * 32), 100, none(), none(), 'chunk_text', jfloat(0.5), 'default', 'table')
    print(f'Visible rows after: {after.length()}')

## 12. `AilakeNative.evolveSchema` — metadata-only schema change (real call)

Also a real call now — adds a new nullable column without rewriting any data files,
then reads the result back through `AilakeNative.scan()` (Fase 11's search+full-row
fetch) to show the new column present as `null` on every pre-existing row.

> **Previously a discovered-but-unfixed limitation, now fixed**: `scan()`/
> `ailake.search(fetch_data=True)` used to build their result columns from each
> physical Parquet file's own schema instead of the table's *current* Iceberg
> schema, so a metadata-only `evolveSchema` addition (no file ever written with the
> new column) never appeared at all — not even as `null` — until a file was
> rewritten (compact/backfill). Root-caused to the single shared
> `ailake_query::scanner::fetch_rows()`, fixed by projecting through the same
> `SchemaFiller` the pointer-search path already used. See `CHANGELOG.md` for the
> full write-up, including a second bug (duplicate/mistyped `embedding` column)
> the fix's first draft introduced and that verifying against real pandas
> conversion caught before it shipped.

In [ ]:
if not AILAKE_PLUGIN_AVAILABLE:
    print('SKIP')
else:
    EVO_PATH = TABLE_PATH.replace('ailake_demo', 'ailake_schema_evo')

    AddColReq = jvm.io.ailake.spark.AilakeNative.AddColReq
    add_cols = to_scala_seq([AddColReq('spark_note', 'string', none())])
    new_schema_id = AilakeNative.evolveSchema(EVO_PATH, 'default', 'table', add_cols, to_scala_seq([]))
    print(f'evolveSchema(add spark_note): new_schema_id={new_schema_id}')

    scan_res = AilakeNative.scan(EVO_PATH, to_float_array([0.0] * 32), 3, 'embedding', none(), 'default', 'table')
    cols = [scan_res.schema().apply(i).name() for i in range(scan_res.schema().length())]
    print(f'columns after evolveSchema (via AilakeNative.scan): {cols}')
    print(f'spark_note present: {"spark_note" in cols}')

## 13. `AilakeNative.search` / `.scan` / `.searchText` / `.compact` (real calls)

The rest of the native surface, all real py4j calls: pointer-only vector search,
search + full-row fetch (no `JOIN`), Tantivy/BM25 full-text search, and file
compaction.

In [ ]:
if not AILAKE_PLUGIN_AVAILABLE:
    print('SKIP')
else:
    import json
    with open(str(pathlib.Path(TABLE_PATH).parent / 'demo_query.json')) as f:
        demo_q = json.load(f)
    query_vec = demo_q['query_vector']

    print('--- AilakeNative.search ---')
    rows = AilakeNative.search(TABLE_PATH, to_float_array(query_vec), 5, none(), none(), 'chunk_text', jfloat(0.5), 'default', 'table')
    for i in range(rows.length()):
        r = rows.apply(i)
        print(f'  row_id={r.rowId():3d}  distance={r.distance():.4f}')

    print()
    print('--- AilakeNative.scan (search + full row, no JOIN) ---')
    scan_res = AilakeNative.scan(TABLE_PATH, to_float_array(query_vec), 3, 'embedding', none(), 'default', 'table')
    print(f'  numRows={scan_res.numRows()}  schema={[scan_res.schema().apply(i).name() for i in range(scan_res.schema().length())]}')

    print()
    print('--- AilakeNative.searchText (FTS) ---')
    fts_path = demo_q['table_paths']['fts']
    text_cols = to_scala_seq(['text'])
    fts_rows = AilakeNative.searchText(fts_path, 'default', 'table', 'machine learning', text_cols, 5, none())
    for i in range(fts_rows.length()):
        r = fts_rows.apply(i)
        print(f'  row_id={r.rowId():3d}  distance={r.distance():.4f}')

    print()
    print('--- AilakeNative.compact ---')
    n = AilakeNative.compact(TABLE_PATH, 'default', 'table', 1, 128 * 1024 * 1024, 20, False)
    print(f'  files_compacted: {n}')

## 13B. `AilakeNative.createTable` — provision an empty table (real call)

Unlike `writeBatch`/`writeBatchMulti`, `createTable` takes only `String`/`Int` arguments
— no `Seq`, `Option`, or boxed `Float`, so it doesn't hit the py4j marshalling limits
noted in §14 below. Added in PR #104; not yet wired into `AilakeCatalog`'s SQL
`CREATE TABLE` DDL (see `docs/specs/JVM_PLUGINS.md`) — this raw py4j call is the only
way to reach it from Spark today.

In [ ]:
if not AILAKE_PLUGIN_AVAILABLE:
    print('SKIP')
else:
    import shutil
    CREATE_PATH = str(pathlib.Path(TABLE_PATH).parent / 'nb_spark_create')
    shutil.rmtree(CREATE_PATH, ignore_errors=True)

    ok = AilakeNative.createTable(CREATE_PATH, 'default', 'table', 'embedding', 32, 'cosine', 'f16', 2)
    print(f'AilakeNative.createTable: {ok}')

    try:
        AilakeNative.createTable(CREATE_PATH, 'default', 'table', 'embedding', 32, 'cosine', 'f16', 2)
        print('ERROR: second call should have raised')
    except Exception as e:
        print(f'Second call raises (table exists): {str(e)[:120]}...')

## 14. `AilakeNative.writeBatch` — write from Spark (real call)

`writeBatch` has 21 positional parameters (Scala default arguments aren't callable
positionally-optional over py4j — every parameter must be supplied explicitly).
`writeBatchMulti` (multi-column/multimodal writes) and `searchMultimodal` (cross-modal
RRF) are **not** demonstrated here: both take a `Seq` of tuples/case-class instances
containing a `Float` field, and py4j always marshals a Python `float` as
`java.lang.Double` with no client-side way to force a real `java.lang.Float` box —
the call reaches the JVM and fails Scala's `unboxToFloat` at the tuple-unpacking site.
This is a genuine limitation of raw py4j reflection into rich Scala signatures, not
a bug in the plugin — production Scala/Java Spark jobs call these directly with no
issue (see `implicits.AilakeSession.ailakeSearchMultimodal`, `CLAUDE.md` Fase 8/12).

In [ ]:
if not AILAKE_PLUGIN_AVAILABLE:
    print('SKIP')
else:
    import shutil
    WRITE_PATH = str(pathlib.Path(TABLE_PATH).parent / 'nb_spark_write')
    shutil.rmtree(WRITE_PATH, ignore_errors=True)

    def jlong(v): return jvm.java.lang.Long(v)
    def scala_float_seq(pylist):
        jlist = jvm.java.util.ArrayList()
        for x in pylist:
            jlist.add(jfloat(x))
        return jvm.scala.collection.JavaConverters.asScalaBuffer(jlist).toSeq()
    def empty_scala_map():
        jmap = jvm.java.util.HashMap()
        return jvm.scala.collection.JavaConverters.mapAsScalaMapConverter(jmap).asScala().toMap(jvm.scala.Predef.conforms())

    ids = to_scala_seq([jlong(0), jlong(1), jlong(2)])
    row_embs = to_scala_seq([
        scala_float_seq([0.1, 0.2, 0.3]),
        scala_float_seq([0.4, 0.5, 0.6]),
        scala_float_seq([0.7, 0.8, 0.9]),
    ])
    snap = AilakeNative.writeBatch(
        WRITE_PATH, 'default', 'table', 'embedding', 3, 'cosine', 'f16',
        ids, row_embs,
        none(), none(), none(), to_scala_seq([]), 2, to_scala_seq([]), 'default',
        none(), none(), False, False,
        empty_scala_map(),
    )
    print(f'writeBatch: snapshot_id={snap}')

    readback = AilakeNative.search(WRITE_PATH, to_float_array([0.1, 0.2, 0.3]), 3, none(), none(), 'chunk_text', jfloat(0.5), 'default', 'table')
    for i in range(readback.length()):
        r = readback.apply(i)
        print(f'  readback row_id={r.rowId()}  distance={r.distance():.4f}')

In [ ]:
spark.stop()
print('SparkSession stopped.')

## Key takeaway

Spark reads AI-Lake tables as **standard Iceberg** — the `HadoopCatalog` discovers
the Parquet files from `metadata.json`, and the Iceberg Spec v2 manifests are fully
compatible. No AI-Lake plugin required for tabular queries.

The `embedding` column appears as `binary` — Spark reads the raw F16 bytes but does
not interpret them. Sections 10-14 showed the **ailake Spark plugin**
(`AilakeNative`, via py4j) adding real vector search, FTS, schema evolution, delete,
compact, and write — `deleteWhere`/`evolveSchema`/`search`/`scan`/`searchText`/
`compact`/`writeBatch` all executed for real against this image's built-in plugin
jar. `searchMultimodal`/`writeBatchMulti` are the two methods raw py4j genuinely
can't drive (Scala `Float`-boxing in tuples/case classes) — those need Scala/Java
Spark code, not PySpark reflection (§14 note).

> **Note (dim=32):** The demo fixture uses dim=32 embeddings. Each vector column
> row is 64 bytes (`dim × 2` for F16 storage).